###  **1.3. Basic ETL withh the DataFrame API**

This demonstration will walk through common ETL operations using the Flights dataset. We'll cover data loaging, cleaning, tranformation, and analysis usgin the DataFrame API.

####  **Objectives**
* Implement common ETL operations using Spark DataFrames
* Handle data cleaning and type conversion
* Create derived features through transformations

NOTE: this section is optional and should be taught at start to give intro to DataFrame API

#### **A. Classroom Setup**


In [ ]:
from pyspark.sql import SparkSession

# Create a the first SparkSession
spark = SparkSession.builder.appName("Basic ETL").getOrCreate()

#### **B. Data Loading and Inspection**

First, let's load and inspect the flight data.

In [ ]:
# Read the flights data from a CSV file and print the first 5 rows
flights_df = spark.read.csv("../resources/DelayedFlights.csv", header=True, inferSchema=True)
flights_df.printSchema()

In [ ]:
# Visually inspect a subset of the data
# display(flights_df.limit(10)) # Display is a Databricks-specific function
flights_df.show(10)

In [ ]:
# Let's remove columns we dont need, remember "filter early, filter often"
flights_required_cols_df = flights_df.select(
    "Year", 
    "Month", 
    "DayofMonth", 
    "DepTime", 
    "FlightNum",
    "ActualElapsedTime",
    "CRSElapsedTime",
    "ArrDelay")

In [ ]:
# Get a count of the source data records
initial_count = flights_required_cols_df.count()

print(f"Source data has {initial_count} records")

In [ ]:
# Let's examine the data for invalid values, these can include null or invalid values for string columns "ArrDelay", "ActualElapsedTime", 
# "DepTime" which we inted on performing mathematical operations on, we can use Spark SQL COUNT_IF function to perform the analysis

# Register the DataFrame as a temporary SQL table with cast columns
flights_required_cols_df \
    .selectExpr(
        "Year", 
        "Month", 
        "DayofMonth",
        "CAST(DepTime AS INT) AS DepTime",
        "FlightNum",
        "CAST(ActualElapsedTime AS INT) AS ActualElapsedTime",
        "CRSELapsedTime",
        "CAST(ArrDelay AS INT) AS ArrDelay"
    ) \
    .createOrReplaceTempView("flights_temp")

####  **Data Cleaning**

The flights data contains some invalid and missing values, lets find them and clean them (in this case we will drop them)

In [ ]:
# To drop rows where any specified columns are null, we can use the na.drop DataFrame method
non_null_flihts_df = flights_required_cols_df.na.drop(
    how="any",
    subset=["CRSElapsedTime"]
)

In [ ]:
from pyspark.sql.functions import col
# Let's remove rows with invalid values for "ArrDelay", "ActualElapsedTime" and "DepTime" columns
flights_with_valid_data_df = non_null_flihts_df.filter(
    (col("ArrDelay").cast("integer").isNotNull()) &
    (col("ActualElapsedTime").cast("integer").isNotNull()) &
    (col("DepTime").cast("integer").isNotNull())
)

In [ ]:
# Now that we know "ArrDelay" and "ActualElapsedTime" contains integer values only, lets cast them from strings to integers (replacing the existing columns)
clean_flights_df = flights_with_valid_data_df \
    .withColumn("ArrDelay", col("ArrDelay").cast("integer")) \
    .withColumn("ActualElapsedTime", col("ActualElapsedTime").cast("integer"))

clean_flights_df.printSchema()

####  **D. Data Enrichment**

Now let's create a useful derived column to categorize delays.

In [ ]:
# Let's start by deriving the "FlightdateTime" column from the "Year", "Month", "DayofMonth" and "DepTime" columns, then drop the constituent columns
from pyspark.sql.functions import col, make_timestamp_ntz, lpad, substr, lit

flights_with_datatime_df = clean_flights_df.withColumn(
    "FlightdateTime", 
    make_timestamp_ntz(
        col("Year"), 
        col("Month"), 
        col("DayofMonth"), 
        substr(lpad(col("DepTime"), 4, "0"), lit(1), lit(2)).try_cast("integer"),
        substr(lpad(col("DepTime"), 4, "0"), lit(3), lit(2)).try_cast("integer"),
        lit(0)
    )
).drop("Year", "Month", "DayofMonth", "DepTime")

# Show the resulting DataFrame schema
print(flights_with_datatime_df.show(10))

In [ ]:
# OK now lets derive the "ElapsedTimeDiff" column from the "ActualElapsedTime" and "CRSElapsedTime" columns

from pyspark.sql.functions import col

flights_with_elapsed_time_diff_df = flights_with_datatime_df.withColumn(
    "ElapsedTimeDiff", col("ActualElapsedTime") - col("CRSElapsedTime") 
    ).drop("ActualElapsedTime", "CRSElapsedTime")

print(flights_with_elapsed_time_diff_df.show())

In [ ]:
# Now lets categorize the "ArrDelay" column into categories: "On Time", "Slight Delay", "Moderate Delay, "Serve Delay"

from pyspark.sql.functions import when

enriched_flights_df = flights_with_elapsed_time_diff_df \
    .withColumn("delay_category", when(col("ArrDelay") <= 0, "On Time") \
        .when(col("ArrDelay") <= 15, "Slight Delay")
        .when(col("ArrDelay") <= 60, "Moderate Delay")
        .otherwise("Severe Delay")) \
        .drop("ArrDelay")


In [ ]:
# Displaying the result
enriched_flights_df.show(10)